# Waimea WaveCast — NOAA 51201: tải lịch sử 2005 → hiện tại, lưu CSV theo tháng

**Kế thừa `test.ipynb`**: dùng `requests.Session`, parser định dạng NOAA TXT, `timestamp_utc` và các cột đo sóng; notebook này chuyên cho **NOAA 51201**, không chạy CDIP/Open-Meteo.

**Pipeline:**

1. **Năm đã kết thúc:** tải `51201hYYYY.txt.gz` → lưu `data-txt-gz/YYYY.txt.gz` → giải nén `data-txt/YYYY.txt` → phân tách thành `data-csv/YYYY/January-data-YYYY.csv`, v.v.
2. **Năm hiện tại:** thử lấy các bản lưu trữ theo **tháng** của NOAA (nếu đã công bố); đồng thời đọc `realtime2/51201.txt` để bắt dữ liệu mới. Lưu tệp gốc Real-Time cộng dồn ở `data-txt/YYYY-realtime.txt`.
3. **Chạy lại an toàn:** `.gz` hợp lệ đã có thì **không tải lại**; `.txt` đã có thì **không giải nén lại**; mỗi file CSV tháng **chỉ nối các timestamp chưa tồn tại**, không xóa bản ghi cũ. Các năm đã xử lý được đánh dấu trong manifest để không parse lại vô ích.

> Lưu ý: “Có `.gz`” nghĩa là **đã cache tệp thật đã tải**, không phải dữ liệu mô phỏng (mock data). Tệp NOAA chứa mã thiếu như `999`, `99.0`, `9999.0` hoặc `MM`; trong CSV được chuẩn hóa thành ô trống. Dữ liệu lịch sử và nguồn realtime có thể có **khoảng trống**; notebook không tự tạo số đo giả.

**Nguồn:** [NDBC historical standard meteorological](https://www.ndbc.noaa.gov/data/historical/stdmet/) · [Lịch sử trạm 51201](https://www.ndbc.noaa.gov/station_history.php?station=51201) · [NDBC real-time](https://www.ndbc.noaa.gov/data/realtime2/51201.txt).

## 1. Cài thư viện (chạy 1 lần cho đúng kernel)

In [1]:
%pip -q install requests pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình và tạo cây thư mục

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from calendar import month_name, month_abbr
from collections import Counter
import gzip
import io
import json
import os
import re
import time
import warnings

import pandas as pd
import requests
from IPython.display import display

STATION_ID = "51201"
START_YEAR = 2005
CURRENT_YEAR = datetime.now(timezone.utc).year

# Thư mục waimea_data được tạo theo working directory của kernel Jupyter.
ROOT = Path.cwd() / "waimea_data"
GZ_DIR = ROOT / "data-txt-gz"
TXT_DIR = ROOT / "data-txt"
CSV_DIR = ROOT / "data-csv"
for folder in (GZ_DIR, TXT_DIR, CSV_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Tùy chọn: lần chạy đầu sẽ tải ~21 archive năm, nên NOAA có thể mất một lúc.
INCLUDE_CURRENT_YEAR_ARCHIVES = True
INCLUDE_REALTIME = True
REQUEST_PAUSE_SECONDS = 0.5
MAX_ATTEMPTS = 3
REPROCESS_CACHED_SOURCES = False  # True: parse lại archive, nhưng vẫn không tạo bản ghi trùng

BASE_URL = "https://www.ndbc.noaa.gov"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "WaimeaWaveCast/1.0 (academic research)", "Accept": "*/*"})
MANIFEST_PATH = ROOT / "processed_manifest.json"
RUN_LOG = []

if MANIFEST_PATH.exists():
    try:
        MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    except (ValueError, OSError):
        warnings.warn("Manifest lỗi, sẽ kiểm tra/tái xử lý nguồn đã cache.")
        MANIFEST = {}
else:
    MANIFEST = {}

print("Root:", ROOT)
print(f"Khoảng năm: {START_YEAR}–{CURRENT_YEAR}")
print("Định dạng file: data-txt-gz/YYYY.txt.gz → data-txt/YYYY.txt → data-csv/YYYY/<Month>-data-YYYY.csv")

Root: /run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data
Khoảng năm: 2005–2026
Định dạng file: data-txt-gz/YYYY.txt.gz → data-txt/YYYY.txt → data-csv/YYYY/<Month>-data-YYYY.csv


## 3. Hàm tải tệp, xác minh gzip và giải nén (có cache)

In [3]:
def valid_gzip(path: Path) -> bool:
    """Kiểm tra toàn bộ gzip để tránh cache file tải dở/hỏng."""
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with gzip.open(path, "rb") as handle:
            for _ in iter(lambda: handle.read(1024 * 1024), b""):
                pass
        return True
    except (OSError, EOFError):
        return False


def download_gzip(url: str, target: Path) -> bool:
    """True nếu có tệp hợp lệ (kể cả đã cache). 404 = chưa công bố, bỏ qua."""
    if valid_gzip(target):
        print(f"[CACHE GZ] {target.name} → bỏ qua tải")
        return True

    if target.exists():
        print(f"[CORRUPT] {target.name}: cache không hợp lệ, sẽ tải lại")

    for attempt in range(1, MAX_ATTEMPTS + 1):
        partial = target.with_name(target.name + ".part")
        try:
            with SESSION.get(url, timeout=(12, 90), stream=True) as response:
                if response.status_code == 404:
                    print(f"[404] Chưa có: {url}")
                    return False
                if response.status_code in {429, 500, 502, 503, 504}:
                    if attempt < MAX_ATTEMPTS:
                        pause = min(30, 2 ** attempt)
                        print(f"[RETRY] HTTP {response.status_code}; nghỉ {pause}s")
                        time.sleep(pause)
                        continue
                response.raise_for_status()
                with partial.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=128 * 1024):
                        if chunk:
                            output.write(chunk)
            if not valid_gzip(partial):
                raise ValueError("Phản hồi không phải gzip hợp lệ")
            partial.replace(target)  # atomic nếu cùng filesystem
            print(f"[DOWNLOADED] {target.name} ({target.stat().st_size:,} bytes)")
            if REQUEST_PAUSE_SECONDS:
                time.sleep(REQUEST_PAUSE_SECONDS)
            return True
        except (requests.RequestException, OSError, ValueError) as exc:
            partial.unlink(missing_ok=True)
            if attempt == MAX_ATTEMPTS:
                print(f"[ERROR] {target.name}: {type(exc).__name__}: {exc}")
                return False
            time.sleep(min(30, 2 ** attempt))
    return False


def decompress_cached_gzip(archive: Path, target: Path) -> bool:
    """Chỉ giải nén khi TXT chưa tồn tại; không ghi đè file TXT hợp lệ."""
    if target.exists() and target.stat().st_size > 0:
        print(f"[CACHE TXT] {target.name} → bỏ qua giải nén")
        return True
    temp = target.with_name(target.name + ".part")
    try:
        with gzip.open(archive, "rb") as source, temp.open("wb") as output:
            while chunk := source.read(1024 * 1024):
                output.write(chunk)
        if not temp.exists() or temp.stat().st_size == 0:
            raise ValueError("Dữ liệu giải nén rỗng")
        temp.replace(target)
        print(f"[EXTRACTED] {target.name}")
        return True
    except (OSError, EOFError, ValueError) as exc:
        temp.unlink(missing_ok=True)
        print(f"[ERROR UNZIP] {archive.name}: {exc}")
        return False


def get_text(url: str) -> str | None:
    """Dùng cho realtime: endpoint này trả TXT chứ không phải gzip."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            r = SESSION.get(url, timeout=(12, 60))
            if r.status_code == 404:
                return None
            if r.status_code in {429, 500, 502, 503, 504} and attempt < MAX_ATTEMPTS:
                time.sleep(min(30, 2 ** attempt))
                continue
            r.raise_for_status()
            if not r.text.lstrip().startswith("#"):
                raise ValueError("NOAA không trả về TXT có header")
            return r.text
        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_ATTEMPTS:
                print(f"[ERROR REALTIME] {type(exc).__name__}: {exc}")
                return None
            time.sleep(min(30, 2 ** attempt))
    return None

## 4. Parser NOAA và chuẩn hóa giá trị thiếu

In [4]:
# Đồng nhất schema giữa các phiên bản dữ liệu NOAA (PTDY có thể không có trong file lịch sử).
MEASUREMENT_COLS = [
    "WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD", "PRES",
    "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE",
]
DATE_COLS = ["year", "month", "day", "hour", "minute"]
EXTRA_COLS = ["station_id", "source", "data_type", "source_dataset"]
EXPORT_COLUMNS = ["timestamp_utc"] + DATE_COLS + MEASUREMENT_COLS + EXTRA_COLS

# Sentinel phải xét theo từng cột; 999 ở WDIR và 9999 ở PRES không phải số đo.
MISSING_CODES = {
    "WDIR": [999], "WSPD": [99.0], "GST": [99.0],
    "WVHT": [99.0], "DPD": [99.0], "APD": [99.0], "MWD": [999],
    "PRES": [9999.0], "ATMP": [999.0], "WTMP": [999.0, 99.0],
    "DEWP": [999.0], "VIS": [99.0], "PTDY": [99.0, 999.0],
    "TIDE": [99.0, 99.00],
}


def parse_noaa_text(text: str, dataset: str) -> pd.DataFrame:
    """Đọc cả header lịch sử và realtime, giữ UTC, không tự resample."""
    lines = text.splitlines()
    headers = []
    for line in lines[:10]:
        if not line.lstrip().startswith("#"):
            continue
        tokens = line.strip().lstrip("#").split()
        if (len(tokens) >= 4 and tokens[0].upper() in {"YY", "YYYY", "YR", "YEAR"}
                and tokens[1].upper() == "MM" and tokens[2].upper() == "DD"
                and tokens[3].lower() == "hh"):
            headers.append(tokens)
    if not headers:
        raise ValueError(f"{dataset}: không tìm thấy header NOAA")
    header = headers[0]
    has_minute = len(header) > 4 and header[4] == "mm"

    payload = []
    malformed = 0
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        parts = stripped.split()
        if not re.match(r"^\d{2,4}$", parts[0]):
            continue
        if len(parts) != len(header):
            malformed += 1
            continue
        payload.append(parts)
    if not payload:
        raise ValueError(f"{dataset}: không có dòng hợp lệ (sai cột: {malformed})")
    if malformed:
        warnings.warn(f"{dataset}: bỏ qua {malformed} dòng sai số cột")

    raw = pd.DataFrame(payload, columns=header)
    year = pd.to_numeric(raw[header[0]], errors="coerce")
    year = year.where(year >= 100, year.where(year >= 70, year + 2000) + 0)
    year = year.where(year >= 100, year + 1900)
    # Năm 00–69 → 2000–2069; 70–99 → 1970–1999.
    for name, values in [
        ("year", year), ("month", pd.to_numeric(raw[header[1]], errors="coerce")),
        ("day", pd.to_numeric(raw[header[2]], errors="coerce")),
        ("hour", pd.to_numeric(raw[header[3]], errors="coerce")),
        ("minute", pd.to_numeric(raw[header[4]], errors="coerce") if has_minute else pd.Series(0, index=raw.index)),
    ]:
        raw[name] = values

    timestamp = pd.to_datetime(
        raw[DATE_COLS].rename(columns=dict(zip(DATE_COLS, DATE_COLS))),
        utc=True, errors="coerce",
    )
    raw["timestamp_utc"] = timestamp
    raw = raw.dropna(subset=["timestamp_utc"]).copy()
    if raw.empty:
        raise ValueError(f"{dataset}: không có timestamp hợp lệ")

    for name in MEASUREMENT_COLS:
        if name in raw.columns:
            values = pd.to_numeric(raw[name].replace({"MM": None, "mm": None}), errors="coerce")
            raw[name] = values.mask(values.isin(MISSING_CODES.get(name, [])))
        else:
            raw[name] = float("nan")
    raw[DATE_COLS] = raw[DATE_COLS].astype("int64")
    raw["timestamp_utc"] = raw["timestamp_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    raw["station_id"] = STATION_ID
    raw["source"] = "NOAA_NDBC"
    raw["data_type"] = "observation"
    raw["source_dataset"] = dataset
    raw = raw[EXPORT_COLUMNS].drop_duplicates("timestamp_utc", keep="last")
    return raw.sort_values("timestamp_utc").reset_index(drop=True)


def append_month_csv(frame: pd.DataFrame, year: int, month: int) -> tuple[int, int, Path]:
    """CSV chỉ append bản ghi có timestamp mới; không sửa bản ghi đã lưu."""
    folder = CSV_DIR / str(year)
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / f"{month_name[month]}-data-{year}.csv"
    frame = frame.loc[(frame["year"] == year) & (frame["month"] == month)].copy()
    frame = frame.drop_duplicates("timestamp_utc", keep="last")
    total_input = len(frame)
    if frame.empty:
        return 0, 0, target
    if target.exists() and target.stat().st_size > 0:
        existing = pd.read_csv(target, usecols=["timestamp_utc"], dtype=str)
        seen = set(existing["timestamp_utc"])
        frame = frame.loc[~frame["timestamp_utc"].isin(seen)]
        header = False
    else:
        header = True
    if not frame.empty:
        frame = frame.sort_values("timestamp_utc")
        frame.to_csv(target, mode="a", index=False, header=header,
                     columns=EXPORT_COLUMNS, encoding="utf-8")
    print(f"[{year}-{month:02d}] {target.name}: +{len(frame)} mới / {total_input} đầu vào")
    return len(frame), total_input, target


def process_source(text_file: Path, source_label: str, year_filter: int | None = None) -> dict:
    """Parse TXT, chia tháng; cache manifest giúp bỏ qua cả bước parse ở lần chạy lại."""
    fingerprint = f"{text_file.stat().st_size}:{text_file.stat().st_mtime_ns}"
    entry = MANIFEST.get(source_label, {})
    expected = [ROOT / rel for rel in entry.get("csv_files", [])]
    if (not REPROCESS_CACHED_SOURCES and entry.get("fingerprint") == fingerprint
            and expected and all(p.exists() and p.stat().st_size > 0 for p in expected)):
        print(f"[DONE/CACHE] {source_label}: bỏ qua parse/CSV")
        return {"source": source_label, "status": "cached", "new_rows": 0}

    frame = parse_noaa_text(text_file.read_text(encoding="utf-8", errors="replace"), source_label)
    if year_filter is not None:
        frame = frame.loc[frame["year"] == year_filter].copy()
    total_added = 0
    output_paths = []
    for (year, month), group in frame.groupby(["year", "month"], sort=True):
        added, count, target = append_month_csv(group, int(year), int(month))
        total_added += added
        if count:
            output_paths.append(str(target.relative_to(ROOT)))
    MANIFEST[source_label] = {"fingerprint": fingerprint, "csv_files": output_paths,
                              "updated_utc": datetime.now(timezone.utc).isoformat()}
    manifest_tmp = MANIFEST_PATH.with_suffix(".json.part")
    manifest_tmp.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")
    manifest_tmp.replace(MANIFEST_PATH)
    return {"source": source_label, "status": "processed", "rows": len(frame), "new_rows": total_added}

## 5. Tự kiểm thử parser bằng dòng dữ liệu cùng định dạng file bạn đã gửi

In [5]:
sample = """#YY MM DD hh mm WDIR WSPD GST WVHT DPD APD MWD PRES ATMP WTMP DEWP VIS TIDE
#yr mo dy hr mn degT m/s m/s m sec sec degT hPa degC degC degC mi ft
2025 01 01 00 16 999 99.0 99.0 1.60 14.29 7.00 294 9999.0 999.0 25.6 999.0 99.0 99.00
2025 01 01 00 46 999 99.0 99.0 1.68 13.33 7.04 317 9999.0 999.0 25.6 999.0 99.0 99.00"""
check = parse_noaa_text(sample, "fixture_2025")
assert len(check) == 2
assert pd.isna(check.loc[0, "WSPD"]) and pd.isna(check.loc[0, "ATMP"])
assert check.loc[0, "WVHT"] == 1.60
assert check.loc[0, "timestamp_utc"] == "2025-01-01T00:16:00Z"
print("PASS: parse 2025, mã thiếu 999/99.0, timestamp UTC")
display(check[["timestamp_utc", "WVHT", "DPD", "MWD", "WTMP", "WDIR", "WSPD", "PRES"]])

PASS: parse 2025, mã thiếu 999/99.0, timestamp UTC


,timestamp_utc,WVHT,DPD,MWD,WTMP,WDIR,WSPD,PRES
0,2025-01-01T00:16:00Z,1.60,14.29,294,25.6,NaN,NaN,NaN
1,2025-01-01T00:46:00Z,1.68,13.33,317,25.6,NaN,NaN,NaN


## 6. Năm 2005 đến năm trước hiện tại: tải `.txt.gz` → `.txt` → CSV theo tháng

In [6]:
for year in range(START_YEAR, CURRENT_YEAR):
    url = f"{BASE_URL}/data/historical/stdmet/{STATION_ID}h{year}.txt.gz"
    archive = GZ_DIR / f"{year}.txt.gz"
    txt_file = TXT_DIR / f"{year}.txt"
    print(f"\n========== ARCHIVE {year} ==========")
    if not download_gzip(url, archive):
        RUN_LOG.append({"year": year, "kind": "annual", "status": "unavailable", "new_rows": 0})
        continue
    if not decompress_cached_gzip(archive, txt_file):
        RUN_LOG.append({"year": year, "kind": "annual", "status": "unzip_failed", "new_rows": 0})
        continue
    try:
        outcome = process_source(txt_file, f"annual_{year}", year_filter=year)
        RUN_LOG.append({"year": year, "kind": "annual", **outcome})
    except Exception as exc:
        RUN_LOG.append({"year": year, "kind": "annual", "status": "parse_failed", "error": str(exc), "new_rows": 0})
        print(f"[ERROR] {year}: {type(exc).__name__}: {exc}")
print("\nHoàn tất archive các năm đã kết thúc.")


========== ARCHIVE 2005 ==========
[CACHE GZ] 2005.txt.gz → bỏ qua tải
[CACHE TXT] 2005.txt → bỏ qua giải nén
[ERROR] 2005: ValueError: annual_2005: không tìm thấy header NOAA

========== ARCHIVE 2006 ==========
[CACHE GZ] 2006.txt.gz → bỏ qua tải
[CACHE TXT] 2006.txt → bỏ qua giải nén
[ERROR] 2006: ValueError: annual_2006: không tìm thấy header NOAA

========== ARCHIVE 2007 ==========
[CACHE GZ] 2007.txt.gz → bỏ qua tải
[CACHE TXT] 2007.txt → bỏ qua giải nén
[2007-01] January-data-2007.csv: +0 mới / 710 đầu vào
[2007-02] February-data-2007.csv: +0 mới / 650 đầu vào
[2007-03] March-data-2007.csv: +0 mới / 730 đầu vào
[2007-04] April-data-2007.csv: +0 mới / 250 đầu vào
[2007-05] May-data-2007.csv: +0 mới / 742 đầu vào
[2007-06] June-data-2007.csv: +0 mới / 664 đầu vào
[2007-07] July-data-2007.csv: +0 mới / 728 đầu vào
[2007-08] August-data-2007.csv: +0 mới / 716 đầu vào
[2007-09] September-data-2007.csv: +0 mới / 713 đầu vào
[2007-10] October-data-2007.csv: +0 mới / 742 đầu vào
[2007-1

## 7. Năm hiện tại: tải các `.txt.gz` lưu trữ theo tháng nếu NOAA đã công bố

Ví dụ tháng 1/2026: `https://www.ndbc.noaa.gov/data/stdmet/Jan/5120112026.txt.gz`. NOAA **không nhất thiết công bố đủ 12 tháng ngay lập tức**. File tháng mới được tải khi NOAA phát hành; file tháng đã cache không tải lại.

In [7]:
if INCLUDE_CURRENT_YEAR_ARCHIVES:
    for month in range(1, datetime.now(timezone.utc).month + 1):
        short = month_abbr[month]
        # NOAA: {station}{month_numeric}{year}.txt.gz, trong /data/stdmet/{Jan|Feb|...}/
        url = f"{BASE_URL}/data/stdmet/{short}/{STATION_ID}{month}{CURRENT_YEAR}.txt.gz"
        archive = GZ_DIR / f"{CURRENT_YEAR}-{month:02d}.txt.gz"
        txt_file = TXT_DIR / f"{CURRENT_YEAR}-{month:02d}.txt"
        print(f"\n========== {CURRENT_YEAR}-{month:02d} ==========")
        if not download_gzip(url, archive):
            RUN_LOG.append({"year": CURRENT_YEAR, "month": month, "kind": "monthly", "status": "unavailable", "new_rows": 0})
            continue
        if not decompress_cached_gzip(archive, txt_file):
            RUN_LOG.append({"year": CURRENT_YEAR, "month": month, "kind": "monthly", "status": "unzip_failed", "new_rows": 0})
            continue
        try:
            outcome = process_source(txt_file, f"monthly_{CURRENT_YEAR}_{month:02d}", year_filter=CURRENT_YEAR)
            RUN_LOG.append({"year": CURRENT_YEAR, "month": month, "kind": "monthly", **outcome})
        except Exception as exc:
            RUN_LOG.append({"year": CURRENT_YEAR, "month": month, "kind": "monthly", "status": "parse_failed", "error": str(exc), "new_rows": 0})
            print(f"[ERROR] {CURRENT_YEAR}-{month:02d}: {type(exc).__name__}: {exc}")


========== 2026-01 ==========
[CACHE GZ] 2026-01.txt.gz → bỏ qua tải
[CACHE TXT] 2026-01.txt → bỏ qua giải nén


[2026-01] January-data-2026.csv: +0 mới / 1486 đầu vào

========== 2026-02 ==========
[CACHE GZ] 2026-02.txt.gz → bỏ qua tải
[CACHE TXT] 2026-02.txt → bỏ qua giải nén
[2026-02] February-data-2026.csv: +0 mới / 1174 đầu vào

========== 2026-03 ==========
[404] Chưa có: https://www.ndbc.noaa.gov/data/stdmet/Mar/5120132026.txt.gz

========== 2026-04 ==========
[404] Chưa có: https://www.ndbc.noaa.gov/data/stdmet/Apr/5120142026.txt.gz

========== 2026-05 ==========
[CACHE GZ] 2026-05.txt.gz → bỏ qua tải
[CACHE TXT] 2026-05.txt → bỏ qua giải nén
[2026-05] May-data-2026.csv: +0 mới / 1177 đầu vào

========== 2026-06 ==========
[CACHE GZ] 2026-06.txt.gz → bỏ qua tải
[CACHE TXT] 2026-06.txt → bỏ qua giải nén
[2026-06] June-data-2026.csv: +0 mới / 1439 đầu vào

========== 2026-07 ==========
[CACHE GZ] 2026-07.txt.gz → bỏ qua tải
[CACHE TXT] 2026-07.txt → bỏ qua giải nén
[2026-07] July-data-2026.csv: +0 mới / 1488 đầu vào

========== 2026-08 ==========
[CACHE GZ] 2026-08.txt.gz → bỏ qua tải
[CAC

## 8. Năm hiện tại: thêm bản ghi Real-Time mới mà không ghi trùng

`realtime2/51201.txt` chỉ chứa cửa sổ gần nhất (~45 ngày), nên phải **lưu lại ngay các bản ghi mới**. TXT hiện tại được nối thêm các dòng timestamp chưa có; CSV theo tháng được cập nhật theo cùng khóa. Không dùng `to_csv()` ghi đè cả tháng.

In [8]:
def append_realtime_raw(text: str, target: Path, year: int) -> int:
    """Lưu raw TXT theo năm dạng append-only, header chỉ một lần."""
    lines = text.splitlines()
    header_lines = [line for line in lines[:3] if line.startswith("#")]
    new_data = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and int(parts[0]) == year:
            new_data.append(line.strip())
    if not new_data:
        return 0

    if target.exists() and target.stat().st_size > 0:
        old_lines = target.read_text(encoding="utf-8").splitlines()
        old_header = next((line for line in old_lines if line.startswith("#YY") or line.startswith("#YYYY")), None)
        fresh_header = next((line for line in header_lines if line.startswith("#YY") or line.startswith("#YYYY")), None)
        if old_header and fresh_header and old_header.split() != fresh_header.split():
            raise ValueError("Schema NOAA Real-Time đã thay đổi; cần file raw riêng, không nối sai số cột")
        seen = {tuple(line.split()[:5]) for line in old_lines
                if line and not line.startswith("#") and len(line.split()) >= 5}
        unique = [line for line in new_data if tuple(line.split()[:5]) not in seen]
        if unique:
            with target.open("a", encoding="utf-8") as fh:
                fh.write("\n".join(unique) + "\n")
    else:
        seen = set()
        unique = []
        for line in new_data:
            key = tuple(line.split()[:5])
            if key not in seen:
                seen.add(key)
                unique.append(line)
        with target.open("w", encoding="utf-8") as fh:
            fh.write("\n".join(header_lines + unique) + "\n")
    return len(unique)


if INCLUDE_REALTIME:
    realtime_url = f"{BASE_URL}/data/realtime2/{STATION_ID}.txt"
    print("\n========== REAL-TIME ==========")
    realtime_text = get_text(realtime_url)
    if realtime_text is None:
        RUN_LOG.append({"year": CURRENT_YEAR, "kind": "realtime", "status": "unavailable", "new_rows": 0})
    else:
        try:
            raw_target = TXT_DIR / f"{CURRENT_YEAR}-realtime.txt"
            new_raw = append_realtime_raw(realtime_text, raw_target, CURRENT_YEAR)
            current_frame = parse_noaa_text(realtime_text, "realtime_45day")
            current_frame = current_frame.loc[current_frame["year"] == CURRENT_YEAR]
            added = 0
            for (year, month), group in current_frame.groupby(["year", "month"], sort=True):
                new_count, _, _ = append_month_csv(group, int(year), int(month))
                added += new_count
            RUN_LOG.append({"year": CURRENT_YEAR, "kind": "realtime", "status": "ok", "new_rows": added})
            print(f"Real-time: +{new_raw} dòng raw, +{added} dòng CSV (không trùng).")
        except Exception as exc:
            print(f"[ERROR REALTIME PROCESS] {type(exc).__name__}: {exc}")
            RUN_LOG.append({"year": CURRENT_YEAR, "kind": "realtime", "status": "parse_failed", "error": str(exc), "new_rows": 0})


========== REAL-TIME ==========
[2026-09] September-data-2026.csv: +0 mới / 379 đầu vào
[2026-10] October-data-2026.csv: +0 mới / 1475 đầu vào
[2026-11] November-data-2026.csv: +99 mới / 301 đầu vào
Real-time: +99 dòng raw, +99 dòng CSV (không trùng).


## 9. Kiểm tra cây thư mục, thống kê dữ liệu, log

Chạy lại toàn bộ notebook: các file cũ không tải lại/không giải nén lại; tháng không xuất hiện timestamp mới thì CSV **không bị ghi lại**. Nếu NOAA trả 403/404 hoặc dữ liệu chưa phát hành, notebook ghi log và tiếp tục.

In [9]:
log_frame = pd.DataFrame(RUN_LOG)
if not log_frame.empty:
    display(log_frame.tail(30))
    print("Tổng bản ghi CSV mới thêm trong lượt chạy:", int(log_frame["new_rows"].fillna(0).sum()))

csv_files = sorted(CSV_DIR.glob("*/*-data-*.csv"))
summary_rows = []
for path in csv_files:
    t = pd.read_csv(path, usecols=["timestamp_utc"], dtype=str)
    summary_rows.append({
        "year": int(path.parent.name), "file": str(path.relative_to(ROOT)),
        "records": len(t), "unique_timestamps": t["timestamp_utc"].nunique(),
        "start": t["timestamp_utc"].min(), "end": t["timestamp_utc"].max(),
    })
summary = pd.DataFrame(summary_rows)
if not summary.empty:
    print("Số file CSV tháng:", len(summary))
    print("Tổng dòng:", int(summary.records.sum()))
    assert (summary.records == summary.unique_timestamps).all(), "Phát hiện CSV có timestamp trùng"
    display(summary.groupby("year", as_index=False).agg(months=("file", "size"), records=("records", "sum")))
    display(summary.tail(15))
else:
    print("Chưa có CSV; kiểm tra log mạng hoặc archive.")

print("\nVí dụ cây thư mục:")
for directory in (GZ_DIR, TXT_DIR):
    print(directory.relative_to(ROOT), ':', len(list(directory.iterdir())), 'files')
print("CSV root:", CSV_DIR)
print("Manifest:", MANIFEST_PATH)

,year,kind,status,error,new_rows,source,rows,month
3,2008,annual,processed,NaN,0,annual_2008,13025.0,NaN
4,2009,annual,processed,NaN,0,annual_2009,16807.0,NaN
5,2010,annual,processed,NaN,0,annual_2010,15722.0,NaN
6,2011,annual,processed,NaN,0,annual_2011,16542.0,NaN
7,2012,annual,processed,NaN,0,annual_2012,16614.0,NaN
8,2013,annual,processed,NaN,0,annual_2013,15855.0,NaN
9,2014,annual,processed,NaN,0,annual_2014,17121.0,NaN
10,2015,annual,processed,NaN,0,annual_2015,17079.0,NaN
11,2016,annual,processed,NaN,0,annual_2016,16107.0,NaN
12,2017,annual,processed,NaN,0,annual_2017,16736.0,NaN


Tổng bản ghi CSV mới thêm trong lượt chạy: 99
Số file CSV tháng: 226
Tổng dòng: 295815


,year,months,records
0,2007,12,8063
1,2008,11,13025
2,2009,12,16807
3,2010,12,15722
4,2011,12,16542
5,2012,12,16614
6,2013,12,15855
7,2014,12,17121
8,2015,12,17079
9,2016,12,16107


,year,file,records,unique_timestamps,start,end
211,2025,data-csv/2025/June-data-2025.csv,1440,1440,2025-06-01T00:26:00Z,2025-06-30T23:56:00Z
212,2025,data-csv/2025/March-data-2025.csv,1485,1485,2025-03-01T00:26:00Z,2025-03-31T23:56:00Z
213,2025,data-csv/2025/May-data-2025.csv,1487,1487,2025-05-01T00:26:00Z,2025-05-31T23:56:00Z
214,2025,data-csv/2025/November-data-2025.csv,1440,1440,2025-11-01T00:26:00Z,2025-11-30T23:56:00Z
215,2025,data-csv/2025/October-data-2025.csv,1483,1483,2025-10-01T00:26:00Z,2025-10-31T23:56:00Z
216,2025,data-csv/2025/September-data-2025.csv,1440,1440,2025-09-01T00:26:00Z,2025-09-30T23:56:00Z
217,2026,data-csv/2026/August-data-2026.csv,1488,1488,2026-08-01T00:26:00Z,2026-08-31T23:56:00Z
218,2026,data-csv/2026/February-data-2026.csv,1174,1174,2026-02-01T00:26:00Z,2026-02-25T10:56:00Z
219,2026,data-csv/2026/January-data-2026.csv,1486,1486,2026-01-01T00:26:00Z,2026-01-31T23:56:00Z
220,2026,data-csv/2026/July-data-2026.csv,1488,1488,2026-07-01T00:26:00Z,2026-07-31T23:56:00Z



Ví dụ cây thư mục:
data-txt-gz : 27 files
data-txt : 28 files
CSV root: /run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data/data-csv
Manifest: /run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data/processed_manifest.json


## 10. Lưu ý về quy trình tích lũy

- **Archive năm kết thúc:** file `.gz` được cache; nếu chạy lại không tải lại. Manifest đánh dấu đã chuyển CSV theo tháng.
- **Năm hiện tại:** archive tháng đã xuất hiện sẽ được tải một lần; file tháng mới được NOAA phát hành sẽ được nhận ở lần chạy sau. Real-Time được gọi **mỗi lần chạy**, dòng mới được nối thêm vào raw TXT và CSV; bản ghi trùng `timestamp_utc` không được thêm nữa.
- **Thứ tự ưu tiên:** tải nguồn archive đã công bố trước, rồi mới nạp realtime; bản ghi trùng giữ bản đã có, không tự cập nhật các hiệu chỉnh QC sau đó. Nếu cần thay thế bằng QC, nên có một pipeline đối soát riêng.
- **Giới hạn dữ liệu:** khi NOAA chưa phát hành tháng lưu trữ và dữ liệu đó đã ra khỏi cửa sổ 45 ngày, notebook có thể chưa khôi phục được khoảng trống; sẽ không tạo dữ liệu giả.
- **Timestamp theo UTC:** CSV được phân tháng theo UTC. Nếu cần phân tích theo ngày Hawaii, chuyển múi giờ trong tầng Silver trước khi nhóm theo ngày.
- **Tính liên tục:** notebook chỉ làm mới khi bạn thực thi; muốn 24/7 cần scheduler/cron/container Data Ingestion riêng.
- **Không xóa file nguồn** `.txt.gz` và `.txt`: giữ lại để kiểm tra truy vết (Bronze).